# Spinodoid microstructures as Gaussian Random Fields

**Goal.** Build 2D spinodoid bi-material samples and treat the underlying
field as the Gaussian random field it actually is.

Following Kumar et al., *npj Comput Mater* **6**, 73 (2020), a spinodoid
field is a sum of plane waves with random phases:

$$\varphi(\mathbf{x}) \;=\; \sqrt{\tfrac{2}{N}} \sum_{i=1}^{N} \cos(\beta\, \mathbf{n}_i\!\cdot\!\mathbf{x} + \gamma_i),$$

with $\gamma_i \sim \mathcal{U}[0,2\pi)$, wavenumber $\beta = 2\pi/\lambda$,
and unit directions $\mathbf{n}_i$ drawn from an admissible angular set
$\Theta \subset S^1$ defined by two cone half-angles $(\theta_1, \theta_2)$
(see [`docs/spinodoid.md`](../docs/spinodoid.md) for the full math + figures).

By the CLT, $\varphi$ is asymptotically Gaussian with zero mean and unit
variance — so the spinodoid is just a *directionally-biased* GRF. The
two-phase mask is built by thresholding $\varphi$ at its empirical
$\rho$-quantile, exactly like `make_grf_bimat`.

**Phases.** Same TPU × PLA bi-material as the GRF demo (MPa–mm–ms units):
TPU (soft, $E\!\approx\!30$ MPa, $\nu\!\approx\!0.48$) and PLA (stiff,
$E\!\approx\!3500$ MPa, $\nu\!\approx\!0.36$).

> The helper functions below preview what `fem_sim.make_spinodoid` will be —
> once that lands, the inline definitions in the next cell collapse to a
> single `from fem_sim import make_spinodoid`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from fem_sim import (
    make_uniaxial,
    run_simulation,
    N_GEO_CHANNELS, CH_SOLID, CH_MATID, CH_E, CH_NU, CH_RHO,
    N_BC_CHANNELS,  CH_DISP_MASK, CH_FORCE_MASK, CH_DX, CH_DY, CH_FX, CH_FY,
)
from fem_sim.geometry import _empty, _fill_material   # private helpers, fine inside the notebook

plt.rcParams["figure.dpi"] = 100
OUT = Path("../outputs/notebook_spinodoid")
OUT.mkdir(parents=True, exist_ok=True)

## 0. Inline preview of `make_spinodoid`

Three small functions:

1. `sample_admissible_angles` — rejection-sample directions from the
   $(\theta_1, \theta_2)$ cone-union $\Theta$.
2. `spinodoid_field` — vectorised plane-wave sum (eq. 1).
3. `spinodoid_geo` — thresholds $\varphi$ at the empirical $\rho$-quantile,
   then fills a `(5, H, W)` geometry array using the package's two-phase
   conventions (`mat_id = 1` for phase A, `mat_id = 2` for phase B).

In [ ]:
def sample_admissible_angles(theta1, theta2, n, rng):
    """Rejection-sample n unit-circle angles from Theta = {a : |cos a|>=cos(t1) or |sin a|>=cos(t2)}."""
    if theta1 == 0.0 and theta2 == 0.0:
        raise ValueError("Empty admissible set: at least one of theta1, theta2 must be > 0.")
    cos_t1, cos_t2 = np.cos(theta1), np.cos(theta2)
    out = np.empty(n)
    filled = 0
    while filled < n:
        batch = rng.uniform(0.0, 2.0 * np.pi, size=max(2 * (n - filled), 64))
        accept = (np.abs(np.cos(batch)) >= cos_t1) | (np.abs(np.sin(batch)) >= cos_t2)
        chosen = batch[accept]
        take = min(len(chosen), n - filled)
        out[filled:filled + take] = chosen[:take]
        filled += take
    return out


def spinodoid_field(h, w, theta1_deg, theta2_deg, wavelength, n_waves=1000, seed=0):
    """Plane-wave-sum GRF (eq. 1).  Returns the continuous field, shape (h, w)."""
    rng = np.random.default_rng(seed)
    t1 = np.deg2rad(theta1_deg)
    t2 = np.deg2rad(theta2_deg)
    beta = 2.0 * np.pi / wavelength
    alphas = sample_admissible_angles(t1, t2, n_waves, rng)
    gammas = rng.uniform(0.0, 2.0 * np.pi, size=n_waves)
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    cos_a = np.cos(alphas)[:, None, None]
    sin_a = np.sin(alphas)[:, None, None]
    arg = beta * (cos_a * xx[None] + sin_a * yy[None]) + gammas[:, None, None]
    return np.cos(arg).sum(axis=0) * np.sqrt(2.0 / n_waves)


def spinodoid_geo(
    h, w, *,
    theta1_deg=90.0, theta2_deg=90.0, wavelength=16.0,
    volume_fraction=0.5, n_waves=1000, seed=0,
    E_A=30.0, nu_A=0.48, rho_A=1.2e-9,         # TPU
    E_B=3500.0, nu_B=0.36, rho_B=1.24e-9,      # PLA
):
    """Two-phase spinodoid bi-material geometry.  Returns (5, h, w)."""
    field = spinodoid_field(h, w, theta1_deg, theta2_deg, wavelength, n_waves, seed)
    threshold = np.quantile(field, volume_fraction)
    mask_A = field <= threshold
    geo = _empty(h, w)
    _fill_material(geo, mask_A,  mat_id=1, E=E_A, nu=nu_A, rho=rho_A)
    _fill_material(geo, ~mask_A, mat_id=2, E=E_B, nu=nu_B, rho=rho_B)
    return geo, field

## 1. One sample — all five geometry channels

Default isotropic case ($\theta_1 = \theta_2 = 90°$): the admissible set is
the entire unit circle, so the field has no preferred direction and the
thresholded mask is a bicontinuous bi-material network.

In [ ]:
geo, _ = spinodoid_geo(
    h=64, w=128,
    theta1_deg=90, theta2_deg=90,   # isotropic
    wavelength=18.0,
    volume_fraction=0.5,
    n_waves=1500, seed=0,
)
print("geometry shape:", geo.shape)
print("material IDs present:", np.unique(geo[CH_MATID]))
print("phase-A area fraction:", float((geo[CH_MATID] == 1).mean()))

titles = ["solid_mask", "material_id", "E (MPa)", "nu", "rho (Mg/mm^3)"]
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for i, ax in enumerate(axes):
    im = ax.imshow(geo[i], origin="lower", cmap="viridis")
    ax.set_title(titles[i]); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 2. The Gaussian-random-field nature

Three diagnostics on the underlying continuous field $\varphi$:

* **Spatial map** — smooth blob-y field (zero mean, unit variance by
  construction).
* **Histogram** — close to $\mathcal{N}(0, 1)$ for $N \gg 1$. The vertical
  line marks the empirical quantile $q_\rho$ used for thresholding.
* **Radial power spectrum** — concentrated around the design wavenumber
  $\beta = 2\pi/\lambda$, since every wave has the same magnitude $\beta$
  in eq. 1 (the *only* random thing about the wave vectors is their
  direction).

In [ ]:
WL = 18.0
_, phi = spinodoid_geo(h=192, w=192, theta1_deg=90, theta2_deg=90,
                       wavelength=WL, volume_fraction=0.5,
                       n_waves=2000, seed=11)
rho = 0.5
q = np.quantile(phi, rho)

# Radial power spectrum.
F = np.fft.fftshift(np.fft.fft2(phi))
P = (np.abs(F) ** 2) / phi.size
ky = np.fft.fftshift(np.fft.fftfreq(phi.shape[0])) * 2 * np.pi
kx = np.fft.fftshift(np.fft.fftfreq(phi.shape[1])) * 2 * np.pi
KX, KY = np.meshgrid(kx, ky)
Krad = np.sqrt(KX**2 + KY**2)
bins = np.linspace(0, Krad.max(), 60)
centers = 0.5 * (bins[1:] + bins[:-1])
radial = np.array([P[(Krad >= bins[i]) & (Krad < bins[i+1])].mean() for i in range(len(bins)-1)])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
im0 = axes[0].imshow(phi, origin="lower", cmap="RdBu_r")
axes[0].set_title(r"continuous field $\varphi(x)$"); axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].hist(phi.ravel(), bins=70, color="0.4", density=True, label="empirical")
xs = np.linspace(-4, 4, 200)
axes[1].plot(xs, np.exp(-xs**2/2)/np.sqrt(2*np.pi), "C2", lw=1.5, label=r"$\mathcal{N}(0,1)$")
axes[1].axvline(q, color="C3", lw=2, label=fr"$q_\rho$ ($\rho={rho}$)")
axes[1].set_title("field histogram"); axes[1].set_xlabel(r"$\varphi$"); axes[1].legend(fontsize=8)

axes[2].plot(centers, radial, color="C0")
axes[2].axvline(2*np.pi/WL, color="C3", ls="--", lw=1.2, label=fr"$\beta=2\pi/\lambda$")
axes[2].set_xlabel(r"$|k|$"); axes[2].set_ylabel("radial power")
axes[2].set_title("power spectrum (isotropic)"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Anisotropy from $(\theta_1, \theta_2)$

The cone half-angles around $\pm x$ ($\theta_1$) and $\pm y$ ($\theta_2$)
control the directional bias of the wave-vector ensemble. Notable presets:

| preset       | $\theta_1$ | $\theta_2$ | morphology                           |
|--------------|-----------:|-----------:|--------------------------------------|
| isotropic    |        90° |        90° | bicontinuous blobs                   |
| lamellar-x   |        15° |         0° | waves along $\pm x$ → vertical stripes |
| lamellar-y   |         0° |        15° | waves along $\pm y$ → horizontal stripes |
| cubic        |        20° |        20° | mutually orthogonal stripes (cross-hatch) |
| oblique      |        25° |        55° | unequal lamellar superposition       |

In [ ]:
presets = [
    ("isotropic",  90, 90),
    ("lamellar-x", 15,  0),
    ("lamellar-y",  0, 15),
    ("cubic",      20, 20),
    ("oblique",    25, 55),
]
fig, axes = plt.subplots(1, len(presets), figsize=(15, 3.2))
for ax, (name, t1, t2) in zip(axes, presets):
    g, _ = spinodoid_geo(h=128, w=128, theta1_deg=t1, theta2_deg=t2,
                          wavelength=18.0, volume_fraction=0.5,
                          n_waves=1500, seed=7)
    ax.imshow(g[CH_MATID], origin="lower", cmap="coolwarm", vmin=1, vmax=2,
              interpolation="nearest")
    ax.set_title(fr"{name}  ($\theta_1$={t1}°, $\theta_2$={t2}°)", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. The two physical knobs: wavelength × volume fraction (isotropic)

Rows vary the characteristic wavelength $\lambda$ (smaller = finer features).
Columns vary the target volume fraction $\rho$ of phase A (TPU).

In [ ]:
wls = [10.0, 18.0, 32.0]
vfs = [0.3, 0.5, 0.7]
fig, axes = plt.subplots(len(wls), len(vfs), figsize=(11, 11))
for i, wl in enumerate(wls):
    for j, vf in enumerate(vfs):
        g, _ = spinodoid_geo(h=128, w=128, theta1_deg=90, theta2_deg=90,
                              wavelength=wl, volume_fraction=vf,
                              n_waves=1500, seed=21)
        axes[i, j].imshow(g[CH_MATID], origin="lower", cmap="coolwarm",
                          vmin=1, vmax=2, interpolation="nearest")
        axes[i, j].set_title(fr"$\lambda$={wl}, $\rho$={vf}", fontsize=10)
        axes[i, j].axis("off")
plt.tight_layout(); plt.show()

## 5. Attach a load case and solve

Square sample, lamellar-x morphology, uniaxial $x$-tension via
`make_uniaxial`. Quasi-static ramp in 5 steps with the JAX-FEM backend.

In [ ]:
geo, _ = spinodoid_geo(h=48, w=48, theta1_deg=15, theta2_deg=0,
                        wavelength=10.0, volume_fraction=0.5,
                        n_waves=1500, seed=3)
bc = make_uniaxial(geo, disp_mag=0.3, direction="x")
print("geo:", geo.shape, "  bc:", bc.shape)

fig, axes = plt.subplots(1, 7, figsize=(20, 3))
axes[0].imshow(geo[CH_MATID], origin="lower", cmap="coolwarm",
               vmin=1, vmax=2, interpolation="nearest")
axes[0].set_title("material_id"); axes[0].axis("off")
for i, name in enumerate(["disp_mask", "force_mask", "dx", "dy", "fx", "fy"]):
    im = axes[i+1].imshow(bc[i], origin="lower", cmap="RdBu_r")
    axes[i+1].set_title(name); axes[i+1].axis("off")
    plt.colorbar(im, ax=axes[i+1], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

In [ ]:
sample = run_simulation(geo, bc, steps=5,
                        run_dir=OUT / "single_run", backend="jaxfem")
print("fields shape:", sample.fields.shape)

final = sample.fields[-1]
field_titles = ["ux (mm)", "uy (mm)", "sxx (MPa)", "syy (MPa)", "sxy (MPa)"]
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for i, ax in enumerate(axes):
    im = ax.imshow(final[i], origin="lower", cmap="RdBu_r")
    ax.set_title(field_titles[i]); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 6. Next steps

Once `make_spinodoid` lands in `fem_sim.geometry` and is registered as
`"spinodoid"` in `campaign._GEO_GENERATORS`, the inline helpers in this
notebook collapse to a single import, and you can drive a full campaign
from a YAML config that reuses the same materials library as
`configs/grf_bimat_campaign.yaml`:

```yaml
geometries:
  - {type: spinodoid, h: 64, w: 64,
     theta1_deg: 90, theta2_deg: 90,        # isotropic
     wavelength: 18.0, volume_fraction: 0.5,
     n_waves: 1500, seed: 0,
     materials: [TPU, PLA]}
  - {type: spinodoid, h: 64, w: 64,
     theta1_deg: 15, theta2_deg: 0,         # lamellar-x
     wavelength: 10.0, volume_fraction: 0.5,
     n_waves: 1500, seed: 1,
     materials: [TPU, PLA]}
```

See [`docs/spinodoid.md`](../docs/spinodoid.md) for the full algorithm,
equations, and reference figures.